# ESM2 full fine tuning


In [1]:
# Dependancies and libraries
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import torch.nn.functional as F
from torch.nn.utils.rnn import pad_sequence
import torch.optim as optim

from transformers import AutoModel, AutoTokenizer, AutoModelForTokenClassification, TrainingArguments, Trainer, DataCollatorForTokenClassification

import esm

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import math

from sklearn.model_selection import GroupShuffleSplit, StratifiedGroupKFold, train_test_split
from sklearn.metrics import accuracy_score, f1_score, recall_score, precision_score, matthews_corrcoef, roc_auc_score, confusion_matrix

from pathlib import Path

In [2]:
# ESM checkpoints
ESM = ['facebook/esm2_t48_15B_UR50D',
        'facebook/esm2_t36_3B_UR50D',
        'facebook/esm2_t33_650M_UR50D',
        'facebook/esm2_t30_150M_UR50D',
        'facebook/esm2_t12_35M_UR50D',
        'facebook/esm2_t6_8M_UR50D']

In [3]:
# Define checkpoint to be used
checkpoint = ESM[5]

In [4]:
# Create tokenizer objects
tokenizer = AutoTokenizer.from_pretrained(checkpoint)

In [5]:
# Import target data
df_target= pd.read_csv("/Users/harry/Documents/Data Science MSc/PROJECT/MScProject/Target_1769.csv")

# Remove extraneous columns
df_target = df_target.iloc[:,0:5]

In [6]:
# Aggreate rows and update format
df_target = df_target.groupby(['Info_protein_id', 'Info_group']).agg(
    sequence=('Info_AA', ''.join), 
    label=('Class', list), 
    position=('Info_pos', list))

In [7]:
df_target.shape
df_target['label'].str.len().agg(['mean','max'])

mean     410.571429
max     1871.000000
Name: label, dtype: float64

In [8]:
# Create a list of sequences
sequences = df_target['sequence'].tolist()

# Instantiate the tokenizer using the AA sequence lists as input to the tokenizer to create tokenized sequences
inputs = tokenizer(
    sequences,
    padding=True,
    truncation=True,
    max_length=1024,
    return_tensors="pt"
)

# Put the model into evaluation mode
model.eval()

# Generate embeddings for the 
with torch.inference_mode():
    outputs = model(**inputs)

In [9]:
# Verify shape of embedding
outputs.last_hidden_state.size()

torch.Size([21, 1024, 320])

In [10]:
# Freeze the weights in the model
for param in model.parameters():
    param.requires_grad = False

In [7]:
# # Create a class column that checks whether the sequence contains a positive or negative epitope and apply a class column for the stratified grouped k fold
# df_lower["class"] = df_lower["label"].apply(lambda x: 1 if 1 in x else -1)

# # Check max length of sequences
# df_lower['label'].str.len().agg(['mean','max'])

Df contains sequences with length > ESM max input length (1024), sliding window will need to be applied once draft complete - same applies for target data

In [8]:
# # Info_group as the grouping variable and Class  / label as the stratification variable.
# X = df_lower.index
# y = df_lower['class']
# groups = df_lower['Info_group']

# # Instantiate GroupShuffleSplit instance to create grouped train/test splits, use 20% of the data for a hold out/test set
# gss = GroupShuffleSplit(n_splits=1, test_size=0.20, random_state=42)

# # Split the data into train/test splits, create indices to be used to assign train/test labels to the df
# train_cv_idx, test_idx = next(gss.split(X, y, groups))

# # Create a train/test column in the dataframe and set the values of the test rows to train or test
# df_lower.loc[test_idx, 'train_test'] = 'test'
# df_lower.loc[train_cv_idx, 'train_test'] = 'train'

# # Update X, y and groups with the remaining train_cv_idx indices to use in Statified Grouped k fold
# X_train, y_train, groups_train = X[train_cv_idx], y[train_cv_idx], groups[train_cv_idx]


In [9]:
# # Split into different folds ensuring stratification accross groups
# sgkf = StratifiedGroupKFold(n_splits=5)

# X_train_df = pd.DataFrame(index=range(0,len(df_lower)))
                                     
# for fold, (train_idx, val_idx) in enumerate(sgkf.split(X, y, groups)):
#     X_train_df.loc[train_idx, f'training_split {fold+1}'] = 1
#     X_train_df.loc[val_idx, f'training_split {fold+1}']= 2

# df_lower = df_lower.merge(X_train_df, left_index=True, right_index=True)

### Sliding window 

- Check whether a sequence is greater than 1024 residues in length
- If > 1024, create copy of sequence
- Create segments

In [6]:
def sliding_window(df, window_size=1024, stride=512):
    """
    Function to create 1024 length splits for proteins >1024 in length, using stride length of 512
    Inserts new splits into a datatable with split suffix
    """
    rows = []
    
    for _, row in df.iterrows():
        seq = row['sequence']
        seq_len = len(seq)
        
        if seq_len <= window_size:
            rows.append(row.copy())
            continue
            
        start = 0
        split = 1
        while start < seq_len:
            # End slicing variable
            end = start + window_size

            # Create copy of the row
            new_row = row.copy()
            # Slice amino acid sequence by start end index
            new_row['sequence'] = seq[start:end]
            
            # Slice columns containing lists
            for col in ['label', 'position']:
                new_row[col] = row[col][start:end]

            # Add split suffix to Info_protein_id
            new_row['Info_protein_id'] = new_row['Info_protein_id'] + '_' + str(split) 
            # Append window to row list 
            rows.append(new_row)
            
            # Stop after the final window
            if end >= seq_len:
                break
                
            start += stride
            split += 1
            
    return pd.DataFrame(rows).reset_index(drop=True)

### Fine Tuning

- Fine tune using higher level data

In [7]:
def preprocess_csv(csv_file):
    preprocessed_df = pd.read_csv(csv_file)

    # Mask n/a values with -100 
    preprocessed_df['Class'] = preprocessed_df['Class'].fillna(-100).astype('int32')
    preprocessed_df['Class'] = preprocessed_df['Class'].replace(-1, 0)
    
    # Sort values before aggregation
    preprocessed_df = preprocessed_df.sort_values(['Info_protein_id', 'Info_pos'])
    
    # Aggregate columns for wide format
    preprocessed_df = preprocessed_df.groupby(['Info_protein_id','Info_group'], as_index=False).agg(
        sequence=('Info_AA', ''.join), 
        label=('Class', list), 
        position=('Info_pos', list))

    # Apply sliding window
    preprocessed_df = sliding_window(preprocessed_df)

    return preprocessed_df
    
# Import higher level data and preprocess to stacked df
df_higher = preprocess_csv('/Users/harry/Documents/Data Science MSc/Higher_1783272.csv')

df_higher

,Info_protein_id,Info_group,sequence,label,position
0,1007216A,190.0,GADDVVDSSKSFVMENFSSYHGTKPGYVDSIQKGIQKPKSGTQGNY...,"[-100, -100, -100, -100, -100, -100, -100, -10...","[1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14..."
1,1106184A,270.0,MKNYLSFGMFALLFALTFGTVNSVQAIAGPEWLLDRPSVNNSQLVV...,"[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, -100, ...","[1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14..."
2,A26297,236.0,MAKNNTNRHYSLRKLKKGTASVAVALSVIGAGLVVNTNEVSARVFP...,"[-100, -100, -100, -100, -100, -100, -100, -10...","[1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14..."
3,A32192,60.0,VKNNLRYGIRKHKLGAASVFLGTMIVVGMGQDKEAAASEQKTTTVE...,"[-100, -100, -100, -100, -100, -100, -100, -10...","[1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14..."
4,A60328,393.0,MNQKIVVISSFYMLGAHSFSKAVYHNDRSVKLMKRIDINHQAQRFS...,"[-100, -100, -100, -100, -100, -100, -100, -10...","[1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14..."
...,...,...,...,...,...
513,ZP_03596362.1,516.0,MERLQKVIAHAGVASRRKAEELIKEGKVKVNGKVVTELGVKVTGSD...,"[-100, -100, -100, -100, -100, -100, -100, -10...","[1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14..."
514,ZP_03980798.1,453.0,MDIRFEQVDFTYQPNTPFEQRALFDINMTIKENSYTALVGHTGSGK...,"[-100, -100, -100, -100, -100, -100, -100, -10...","[1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14..."
515,ZP_04069274.1,118.0,MNYMEDSSLDTLSIVNETDFPLYNNYTEPTIAPALIAVAPIAQYLA...,"[-100, -100, -100, -100, -100, -100, -100, -10...","[1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14..."
516,ZP_05686172.1,434.0,MKKLVPLLLALLLLVAACGTGGKQSSDKSNGKLKVVTTNSILYDMA...,"[-100, -100, -100, -100, -100, -100, -100, -10...","[1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14..."


In [8]:
# Creating a train df with 80% of the rows
df_higher_train = df_higher.sample(frac=0.8, random_state=42).reset_index()

# Create validation df with remaining rows
df_higher_val = df_higher.drop(df_higher_train.index).reset_index()

In [9]:
def preprocess(data):
    """
    Preprocess each df row to tokenise the sequences and pad labels to match max length
    """
    inputs = tokenizer(
            data['sequence'],
            padding='max_length',
            truncation=True, 
            max_length=1026,
        )
    
    labels = data['label']

    # Remove first and last special tokens for tokens and attention mask
    input_ids = inputs['input_ids'][1:-1]
    attn_mask = inputs['attention_mask'][1:-1]

    # Pad labels to max length
    labels = labels + ([-100] * (1024 - len(labels)))

    return {'input_ids': input_ids,
            'attention_mask': attn_mask,
            'labels': labels}

preprocessed_train = df_higher_train.apply(preprocess, axis=1)
preprocessed_val = df_higher_val.apply(preprocess, axis=1)

data_collator = DataCollatorForTokenClassification(tokenizer=tokenizer)


In [12]:
# Define Trainer parameters
def compute_metrics(p):

    # Separate logits and labels
    pred, labels = p    
    
    # Return index of higher position (neg or positive residue) per row
    max_pred = np.argmax(pred, axis=-1)

    # Calculate probabilites to be used for AUC
    probs = torch.softmax(torch.tensor(pred), dim=-1)
    probs = probs[:, :, 1]

    # Create mask for unlabelled positions
    mask = labels != -100

    # Calculate metrics
    accuracy = accuracy_score(y_true=labels[mask], y_pred=max_pred[mask])
    recall = recall_score(y_true=labels[mask], y_pred=max_pred[mask])
    precision = precision_score(y_true=labels[mask], y_pred=max_pred[mask])
    f1 = f1_score(y_true=labels[mask], y_pred=max_pred[mask])
    mcc = matthews_corrcoef(y_true=labels[mask], y_pred=max_pred[mask])
    auc = roc_auc_score(y_true=labels[mask], y_score=probs[mask])

    return {'accuracy': accuracy, 'precision': precision, 'recall': recall, 'f1': f1, 'mcc': mcc, 'auc': auc}

model = AutoModelForTokenClassification.from_pretrained(checkpoint)

training_args = TrainingArguments(
    output_dir='/Users/harry/Documents/Data Science MSc/PROJECT/MScProject/fine-tuned-8M',
    # learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=2,
    weight_decay=0.01,
    eval_strategy='epoch',
    save_strategy='epoch',
    logging_strategy='epoch',
    load_best_model_at_end=True,
    metric_for_best_model='auc',
    greater_is_better=True
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=preprocessed_train,
    eval_dataset=preprocessed_val,
    processing_class=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

trainer.train()

# Obtain best model weights
model = trainer.model

# Explicitly save best performing model
trainer.save_model('/Users/harry/Documents/Data Science MSc/PROJECT/MScProject/fine-tuned-8M/best-performing-model')


Loading weights:   0%|          | 0/102 [00:00<?, ?it/s]

[transformers] EsmForTokenClassification LOAD REPORT from: facebook/esm2_t6_8M_UR50D
Key                       | Status     | 
--------------------------+------------+-
lm_head.layer_norm.weight | UNEXPECTED | 
lm_head.bias              | UNEXPECTED | 
lm_head.dense.bias        | UNEXPECTED | 
lm_head.dense.weight      | UNEXPECTED | 
lm_head.layer_norm.bias   | UNEXPECTED | 
classifier.weight         | MISSING    | 
classifier.bias           | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
/opt/anaconda3/lib/python3.13/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


Epoch,Training Loss,Validation Loss,Accuracy,Precision,Recall,F1,Mcc,Auc
1,0.585719,0.499867,0.853241,1.000000,0.002812,0.005607,0.048978,0.656662
2,0.557841,0.489070,0.853241,1.000000,0.002812,0.005607,0.048978,0.688456


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/opt/anaconda3/lib/python3.13/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

### Train classification head on embeddings from the fine-tuned model

In [35]:
# Preprocess training data
df_lower = pd.read_csv("/Users/harry/Documents/Data Science MSc/PROJECT/MScProject/Lower_1763.csv")

# Mask n/a values with -100 
df_lower['Class'] = df_lower['Class'].fillna(-100).astype('int32')
df_lower['Class'] = df_lower['Class'].replace(-1, 0)

# Sort values before aggregation
df_lower = df_lower.sort_values(['Info_protein_id', 'Info_pos'])

# Aggregate columns for wide format
df_lower = df_lower.groupby(['Info_protein_id','Info_group', 'Info_split'], as_index=False).agg(
    sequence=('Info_AA', ''.join), 
    label=('Class', list), 
    position=('Info_pos', list))

# Apply sliding window
df_lower = sliding_window(df_lower)

# Create train / validation splits using the info split column
cv_1_train = df_lower[df_lower['Info_split'] != 'split_01_20'].reset_index()
cv_1_val = df_lower[df_lower['Info_split'] == 'split_01_20'].reset_index()

cv_2_train = df_lower[df_lower['Info_split'] != 'split_02_20'].reset_index()
cv_2_val = df_lower[df_lower['Info_split'] == 'split_02_20'].reset_index()

cv_3_train = df_lower[df_lower['Info_split'] != 'split_03_20'].reset_index()
cv_3_val = df_lower[df_lower['Info_split'] == 'split_03_20'].reset_index()

cv_4_train = df_lower[df_lower['Info_split'] != 'split_04_20'].reset_index()
cv_4_val = df_lower[df_lower['Info_split'] == 'split_04_20'].reset_index()

cv_5_train = df_lower[df_lower['Info_split'] != 'split_05_20'].reset_index()
cv_5_val = df_lower[df_lower['Info_split'] == 'split_05_20'].reset_index()

device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")

model.to(device)
model.eval()

EsmForTokenClassification(
  (esm): EsmModel(
    (embeddings): EsmEmbeddings(
      (word_embeddings): Embedding(33, 320, padding_idx=1)
      (dropout): Dropout(p=0.0, inplace=False)
    )
    (rotary_embeddings): EsmRotaryEmbedding()
    (encoder): EsmEncoder(
      (layer): ModuleList(
        (0-5): 6 x EsmLayer(
          (attention): EsmAttention(
            (self): EsmSelfAttention(
              (query): Linear(in_features=320, out_features=320, bias=True)
              (key): Linear(in_features=320, out_features=320, bias=True)
              (value): Linear(in_features=320, out_features=320, bias=True)
            )
            (output): EsmSelfOutput(
              (dense): Linear(in_features=320, out_features=320, bias=True)
              (dropout): Dropout(p=0.0, inplace=False)
            )
            (LayerNorm): LayerNorm((320,), eps=1e-05, elementwise_affine=True)
          )
          (intermediate): EsmIntermediate(
            (dense): Linear(in_features=320, out_

In [36]:
# Create custom dataset class - add to separate file and import
class SequenceDataset(torch.utils.data.Dataset):
    
    def __init__(self, df):
        self.protein_id = df['Info_protein_id']
        self.sequence = df['sequence']
        self.position = df['position']
        self.labels = df['label']
        
    def __len__(self):
        return len(self.sequence)
        
    def __getitem__(self, idx):
        return {
            'protein_id': self.protein_id.iloc[idx],
            'sequence': self.sequence.iloc[idx],
            'position': self.position.iloc[idx],
            'label': self.labels.iloc[idx]
        }

def collate_fn(batch):
    """
    Function to create a custom collator to maintain length of items within the batch
    """
    return {
        'protein_id': [x['protein_id'] for x in batch],
        'sequence': [x['sequence'] for x in batch],
        'position': [x['position'] for x in batch],
        'label': [x['label'] for x in batch]
    }

def batch_create(dataset, tokenizer, model): 
    """
    Function to create a DataLoader instance using the the custom collate function
    """
    # Create a DataLoader instance using the cv dataset and the custom collate function
    loader = DataLoader(
        dataset,
        batch_size=16,
        shuffle=False,
        collate_fn=collate_fn
    )
    
    emb_output_list = []
    
    # Loop through each batch of tensors and apply tokenisation to each sequence
    for batch in loader:
        
        inputs = tokenizer(
            batch['sequence'],
            padding=True,
            truncation=True,
            return_tensors='pt'
        )
        
        inputs = {k: v.to(device) for k, v in inputs.items()}
        
        # Freeze model weights and calculate embeddings 
        with torch.no_grad():
            outputs = model.esm(**inputs)
            embeddings = outputs.last_hidden_state[:, 1:-1, :] # Slice embeddings to remove start and end CLS/EOS tokens
    
        # Convert labels to tensors 
        labels = [torch.tensor(x) for x in batch['label']]

        # Pad each label to the size of the largest embedding within the batch
        labels = pad_sequence(labels, batch_first=True, padding_value=-100)
        
        # Append the embeddings, label and masks per batch to an output list 
        emb_output_list.append({
            'embeddings': embeddings,
            'labels': labels})
    
    return emb_output_list

In [37]:
# Create datasets
train_datasets = {
    1:SequenceDataset(cv_1_train),
    2:SequenceDataset(cv_2_train), 
    3:SequenceDataset(cv_3_train),
    4:SequenceDataset(cv_4_train),
    5:SequenceDataset(cv_5_train)    
}

val_datasets = {
    1:SequenceDataset(cv_1_val),
    2:SequenceDataset(cv_2_val), 
    3:SequenceDataset(cv_3_val),
    4:SequenceDataset(cv_4_val),
    5:SequenceDataset(cv_5_val)    
}

# Create train batches
train_loaded = {}

for key, value in train_datasets.items():
    batched = batch_create(value, tokenizer, model)
    train_loaded[key] = batched

# Create val batches
val_loaded = {}

for key, value in val_datasets.items():
    batched = batch_create(value, tokenizer, model) 
    val_loaded[key] = batched

In [38]:
# Classification head
class PerResidueClassifier(nn.Module):
    def __init__(self, in_features=320, hidden_size=128, out_features=2):
        super().__init__()
        self.linear1 = nn.Linear(in_features, hidden_size)
        self.relu = nn.ReLU()
        self.dropout = nn.Dropout(0.2)
        self.linear2 = nn.Linear(hidden_size, out_features)
        
    def forward(self, embeddings):
        x = self.linear1(embeddings)
        x = self.relu(x)
        x = self.dropout(x)
        logits = self.linear2(x)
        
        return logits
        
# Instantiate classifier
clf = PerResidueClassifier()

In [39]:
# Determine frequency of positive vs negative labels to be used to weight the loss function
neg_class = len(df_lower_stacked[df_lower_stacked['Class'] == 0])
pos_class = len(df_lower_stacked[df_lower_stacked['Class'] == 1])

# Total labelled 
total = neg_class + pos_class

# Set the weight as the inverse proportion of the tota
neg_weight = torch.tensor([total/(2*neg_class)])
pos_weight = torch.tensor([total/(2*pos_class)])

# Concat tensors 
weight = torch.cat([neg_weight, pos_weight])


NameError: name 'df_lower_stacked' is not defined

In [40]:
# Checkpoint directory
checkpoint_dir = Path("/Users/harry/Documents/Data Science MSc/PROJECT/MScProject/clf_checkpoint")
checkpoint_dir.mkdir(exist_ok=True)

In [42]:
# Instantiate cross-entropy loss function, ignores positions with mask -100
loss_fcn = nn.CrossEntropyLoss( ignore_index=-100) # weight=weight,

# Dictionaries to store training and validation outputs - update 
training_loss_dict = {}
acc_dict = {}
val_loss_dict = {}
val_acc_dict = {}
f1_val_dict = {}
mcc_val_dict = {}
roc_auc_val_dict = {}

epochs = 20

# Training loop 
for key, fold in train_loaded.items():
    
    training_loss_dict[key] = []
    acc_dict[key] = []
    val_loss_dict[key] = []
    val_acc_dict[key] = []
    f1_val_dict[key] = []
    mcc_val_dict[key] = []
    roc_auc_val_dict[key] = []
    
    # Classifier
    clf = PerResidueClassifier()

    # AdamW optimiser
    optimiser = optim.AdamW(clf.parameters(), lr=0.001)
    
    for epoch in range(1, epochs+1):

        # Put the clf in training mode
        clf.train()

        # Logging per epoch
        running_loss = 0
        train_labels_flat = []
        train_preds = []
        train_probs = []
        
        for batch in fold:
            # print(key, epoch_n, fold_n)
            inputs = batch['embeddings']
            labels = batch['labels']
            
            # Zero model gradients per batch
            optimiser.zero_grad()
        
            # Caluclate logits by passing embeddings through the classifier
            outputs = clf(inputs)

            # Compute loss and gradients        
            loss = loss_fcn(outputs.reshape(-1, 2), labels.reshape(-1))
    
            # Calculate the gradients through the network
            loss.backward()
    
            # Adjust learning weights
            optimiser.step()

            # Add the loss to a running counter
            running_loss += loss.item()

            # Predicted labels, assign pos or negative based on argmax of the pos / neg class logits, flatten
            preds = torch.argmax(outputs, dim=-1).reshape(-1)
            labels_flat = labels.reshape(-1) 
    
            # Append preds and actuals to lists
            train_preds.append(preds)
            train_labels_flat.append(labels_flat)
         
        # Calculate loss per epoch and add to dict
        epoch_loss = running_loss/len(fold)
        training_loss_dict[key].append(epoch_loss)
        
        # Concat the batched tensors in the lists 
        train_preds = torch.cat(train_preds)
        train_labels_flat = torch.cat(train_labels_flat)

        # Mask for accuracy calculation
        accuracy_mask = train_labels_flat != -100

        # Calculate accuracy of non-masked positions and add to dict
        accuracy = accuracy_score(train_labels_flat[accuracy_mask], train_preds[accuracy_mask])
        acc_dict[key].append(accuracy)

        
        """
        ----------------------- Validation loop -----------------------
        """
        
        # Put the classifier in eval mode
        clf.eval()

        val_labels_flat = []
        val_preds = []
        val_probs = []
        running_val_loss = 0

        # Set best val loss for best model recording and set to positive infintiy for first loop
        best_val_loss = float('inf')
        
        with torch.no_grad():
            
            for val_batch in val_loaded[key]:
                val_inputs = val_batch['embeddings']
                val_labels = val_batch['labels']

                val_outputs = clf(val_inputs)

                val_loss = loss_fcn(val_outputs.reshape(-1, 2), val_labels.reshape(-1))
                
                running_val_loss += val_loss.item()

                # Predicted labels, assign pos or negative based on argmax of the neg / pos class logits, flatten
                preds_val = torch.argmax(val_outputs, dim=-1).reshape(-1)
                labels_flat_val = val_labels.reshape(-1)

                # Append preds and actuals to lists
                val_preds.append(preds_val)
                val_labels_flat.append(labels_flat_val)
                
                # Calculate probabilites to be used for AUC
                probs = torch.softmax(val_outputs, dim=-1)
                pos_probs_flat = probs[:, :, 1].reshape(-1)
                val_probs.append(pos_probs_flat)
        
        # Calculate loss per epoch and append to dict
        epoch_val_loss = running_val_loss/len(val_loaded[key])
        val_loss_dict[key].append(epoch_val_loss)

        # Concat the batched tensors in the lists 
        val_preds = torch.cat(val_preds)
        val_labels_flat = torch.cat(val_labels_flat) 
        val_probs = torch.cat(val_probs)
        
        # Create mask to ignore unlabelled poisitons 
        mask_val = val_labels_flat != -100

        # Calculate accuracy of non-masked positions and append to dict
        accuracy_val = accuracy_score(val_labels_flat[mask_val], val_preds[mask_val])
        val_acc_dict[key].append(accuracy_val)

        # Calculate F1 score and append to dict
        f1_val = f1_score(val_labels_flat[mask_val], val_preds[mask_val])
        f1_val_dict[key].append(f1_val)

        # Calculate MCC and append to dict
        mcc_val = matthews_corrcoef(val_labels_flat[mask_val], val_preds[mask_val])
        mcc_val_dict[key].append(mcc_val)

        # Calculate ROC/AUC score
        roc_auc_val = roc_auc_score(val_labels_flat[mask_val], val_probs[mask_val])
        roc_auc_val_dict[key].append(roc_auc_val)

        # Save best model
        if epoch_val_loss < best_val_loss:
            best_val_loss = epoch_val_loss
            torch.save(clf.state_dict(), checkpoint_dir / 'best_clf_fine_tuned_ESM8M.pt')

RuntimeError: Tensor for argument weight is on cpu but expected on mps

### Predictions on test data

In [45]:
# Preprocess test set and generate embeddings using ESM2 as standard model
test_df = preprocess_csv('/Users/harry/Documents/Data Science MSc/PROJECT/MScProject/Target_1769.csv')

# Tokenise and preprocess labels
preprocessed_test = test_df.apply(preprocess, axis=1)

In [46]:
# Load full fine tuned model (to save on completing full fine-tuning)
model = AutoModelForTokenClassification.from_pretrained('/Users/harry/Documents/Data Science MSc/PROJECT/MScProject/fine-tuned-8M/best-performing-model')
model.eval()

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

EsmForTokenClassification(
  (esm): EsmModel(
    (embeddings): EsmEmbeddings(
      (word_embeddings): Embedding(33, 320, padding_idx=1)
      (dropout): Dropout(p=0.0, inplace=False)
    )
    (rotary_embeddings): EsmRotaryEmbedding()
    (encoder): EsmEncoder(
      (layer): ModuleList(
        (0-5): 6 x EsmLayer(
          (attention): EsmAttention(
            (self): EsmSelfAttention(
              (query): Linear(in_features=320, out_features=320, bias=True)
              (key): Linear(in_features=320, out_features=320, bias=True)
              (value): Linear(in_features=320, out_features=320, bias=True)
            )
            (output): EsmSelfOutput(
              (dense): Linear(in_features=320, out_features=320, bias=True)
              (dropout): Dropout(p=0.0, inplace=False)
            )
            (LayerNorm): LayerNorm((320,), eps=1e-05, elementwise_affine=True)
          )
          (intermediate): EsmIntermediate(
            (dense): Linear(in_features=320, out_

In [47]:
# Run tokens through the fine-tuned model and compute metrics 
predictions = trainer.predict(preprocessed_test)

/opt/anaconda3/lib/python3.13/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


In [59]:
pd.DataFrame(predictions.metrics, index=np.array(np.arange(1, 2)))

,test_loss,test_accuracy,test_precision,test_recall,test_f1,test_mcc,test_auc,test_runtime,test_samples_per_second,test_steps_per_second
1,0.652153,0.672604,0.594828,0.124324,0.205663,0.148373,0.70285,2.695,8.906,0.742
